# Music Annotation Project V9.0A
## Mask-Authority Transfer Experiment

**Runtime:** CPU for cached detector artefacts and V8.8 execution. Use GPU only if Notebook 06 artefacts must be regenerated separately.

This is a clean, development-only, fail-closed notebook. It isolates **mask authority** while holding the V8.8 correction engine fixed.

### Experimental arms
1. `reviewed_comparator`: reviewer-corrected mask.
2. `raw_yolo`: Notebook 06 draft mask.
3. `confidence_gated_yolo`: raw confidence map gated by the established inherited policy recorded in the Notebook 06 draft audit.
4. `reviewer_corrected_yolo`: reviewed mask paired to the same Notebook 06 draft lineage.

### Non-negotiable contract
- Do not retrain YOLO or V8.8.
- Do not optimise a confidence threshold in this notebook.
- Do not substitute observed-print probabilities for YOLO confidence.
- Run the same page roster through the same frozen V8.8 engine for every arm.
- Keep detector confidence, authoritative mask, M3, SAFE_REMOVE, M5, corrected page and review edits separate.
- Lock masks and corrected outputs before loading clean truth or calculating reviewed-mask comparisons.
- Treat the reviewed mask as the comparator, not automatically as perfect truth.
- Any missing lineage, ambiguous audit, shape mismatch, non-finite confidence, hash mismatch or V8.8 replay failure stops the run.
- This notebook produces research evidence only. It does not authorise automatic mask authority or production output.

Every code cell begins with a provenance comment. `TRANSPLANTED` means the implementation is copied or minimally adapted from the named source. `NEW` means V9.0A orchestration or analysis code written for this notebook.

In [ ]:
# PROVENANCE: TRANSPLANTED from standard project Colab notebook bootstrap; purpose: mount Drive and install declared runtime dependencies.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib pillow

## 1. Configuration and change controls

In [ ]:
# PROVENANCE: NEW for V9.0A; purpose: freeze experiment paths, arm names and fail-closed controls without inventing a YOLO threshold.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
OUTPUT_ROOT = ROOT / 'mask_authority_transfer_v9_0a'
HITL_ROOT = ROOT / 'human_in_the_middle'
DRAFT_AUDIT_DIR = HITL_ROOT / 'draft_audit'
REVIEW_AUDIT_DIR = HITL_ROOT / 'audit'

# V8.8 registered-replay operational lineage.
N53_ROOT = ROOT / 'observed_print_classifier_v8_8_notebook53R3'
N54_ROOT = ROOT / 'm5_remove_exact_equivalence_v8_8c'
NOTEBOOK53_RUN_OVERRIDE = None
NOTEBOOK54_RUN_OVERRIDE = None

# Experiment cohort. Put annotated source pages here or set an exact existing directory.
COHORT_DIR = ROOT / 'mask_authority_transfer_v9_0a' / 'cohort' / 'annotated'
CLEAN_DIR = ROOT / 'source' / 'clean'  # loaded only after prediction/output lock
COHORT_MANIFEST = ROOT / 'mask_authority_transfer_v9_0a' / 'cohort' / 'cohort_manifest.csv'
PAGE_ALLOWLIST = None

# The exact inherited rule must resolve from Notebook 06 audit records.
# This override is deliberately disabled by default. If used, document the prior experiment record and policy ID.
ESTABLISHED_GATE_OVERRIDE = None
ESTABLISHED_GATE_POLICY_ID_OVERRIDE = None
ALLOW_MANUAL_GATE_OVERRIDE = False

ARMS = ['reviewed_comparator','raw_yolo','confidence_gated_yolo','reviewer_corrected_yolo']
SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v9_0a_mask_authority'
OUT = OUTPUT_ROOT / 'runs' / RUN_ID
INPUTS=OUT/'01_inputs'; MASKS=OUT/'02_mask_arms'; ENGINE=OUT/'03_v88_engine'; LOCK=OUT/'04_prediction_lock'; SCORE=OUT/'05_scoring'; VISUAL=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,ENGINE,LOCK,SCORE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Shared integrity helpers

In [ ]:
# PROVENANCE: TRANSPLANTED and consolidated from Notebook 09, Notebook 17 and Notebook 53R3; purpose: deterministic file/array identity, exact input resolution and fail-closed checks.
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    path=Path(path); require(path.exists() and path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def object_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def read_json(path):
    path=Path(path); require(path.exists(),f'Missing JSON: {path}')
    value=json.loads(path.read_text()); require(isinstance(value,dict),f'JSON root must be an object: {path}'); return value

def image_map(folder):
    folder=Path(folder); return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(im is not None,f'Unreadable image: {path}'); return im

def binary(mask): return np.where(mask>0,255,0).astype(np.uint8)

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

## 3. Resolve frozen V8.8 lineage

In [ ]:
# PROVENANCE: TRANSPLANTED and adapted from Notebooks 54, 55 and 57R1; purpose: require a passed Notebook 53R3 registered replay and Notebook 54 exact-equivalence gate.
REQ53=['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','tables/mask_inventory.csv','tables/regional_masks.csv','pages']
REQ54=['notebook54_verdict.json','experiment_identity.json']
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else latest_complete(N53_ROOT,REQ53)
N54=Path(NOTEBOOK54_RUN_OVERRIDE) if NOTEBOOK54_RUN_OVERRIDE else latest_complete(N54_ROOT,REQ54)
require(N53 is not None,'No complete Notebook 53R3 run found.')
require(N54 is not None,'No complete Notebook 54 run found.')
id53=read_json(N53/'experiment_identity.json'); v53=read_json(N53/'stage_f_verdict.json'); v54=read_json(N54/'notebook54_verdict.json')
require(id53.get('version')=='53R3-v8.8b-notebook51R2-regression-compatibility','Unexpected Notebook 53R3 version.')
require(id53.get('mode')=='REGISTERED_REPLAY','Notebook 53R3 must be REGISTERED_REPLAY.')
require(str(v54.get('verdict','')).startswith('PASS_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE'),'Notebook 54 exact-equivalence gate did not pass.')
require(v54.get('first_failed_stage') is None,'Notebook 54 records a failed stage.')
lineage={'notebook53_run':str(N53),'notebook53_identity_sha256':sha256_file(N53/'experiment_identity.json'),'notebook53_verdict_sha256':sha256_file(N53/'stage_f_verdict.json'),'notebook54_run':str(N54),'notebook54_verdict_sha256':sha256_file(N54/'notebook54_verdict.json'),'mode':'REGISTERED_REPLAY'}
atomic_json(AUDIT/'v88_lineage.json',lineage)
print(json.dumps(lineage,indent=2))

## 4. Resolve cohort and Notebook 06 / 07 HITL lineage

In [ ]:
# PROVENANCE: TRANSPLANTED and adapted from Notebook 09 resolve_hitl; purpose: match source pages to draft/review audits by source MD5 rather than filename guesses.
def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def audit_records(folder,label):
    records=[]; failures=[]
    for path in sorted(Path(folder).glob('*.json')):
        try:
            v=json.loads(path.read_text()); require(isinstance(v,dict),'audit root must be object'); v['_audit_path']=str(path); v['_audit_type']=label; records.append(v)
        except Exception as e: failures.append({'audit_type':label,'audit_path':str(path),'error':str(e)})
    return records,failures

def existing_path(record,keys,label):
    for key in keys:
        raw=record.get(key)
        if raw and Path(raw).exists(): return Path(raw),key
    raise RuntimeError(f'{label} audit {record.get("_audit_path")} has no existing path for {keys}')

sources=image_map(COHORT_DIR)
require(sources,f'No cohort images found in {COHORT_DIR}')
if PAGE_ALLOWLIST is not None: sources={k:v for k,v in sources.items() if k in set(PAGE_ALLOWLIST)}
if COHORT_MANIFEST.exists():
    cohort_meta=pd.read_csv(COHORT_MANIFEST,dtype=str); require('page' in cohort_meta and not cohort_meta.page.duplicated().any(),'Invalid cohort manifest page column.')
    require(set(cohort_meta.page)==set(sources),f'Cohort manifest/image roster mismatch: manifest_only={sorted(set(cohort_meta.page)-set(sources))}, images_only={sorted(set(sources)-set(cohort_meta.page))}')
    shutil.copy2(COHORT_MANIFEST,INPUTS/'cohort_manifest.csv')
else:
    cohort_meta=pd.DataFrame({'page':sorted(sources),'source_group':'UNSPECIFIED'}); cohort_meta.to_csv(INPUTS/'cohort_manifest.csv',index=False)

DRAFTS,dfail=audit_records(DRAFT_AUDIT_DIR,'draft'); REVIEWS,rfail=audit_records(REVIEW_AUDIT_DIR,'review')
failures=dfail+rfail
if failures: pd.DataFrame(failures).to_csv(AUDIT/'audit_parse_failures.csv',index=False); raise RuntimeError('Unreadable HITL audit records found.')

resolved=[]
for page,source in sources.items():
    smd5=md5_file(source)
    drafts=[r for r in DRAFTS if str(r.get('source_md5','')).lower()==smd5]
    reviews=[r for r in REVIEWS if str(r.get('source_md5',r.get('source_md5sum',''))).lower()==smd5]
    require(len(drafts)==1,f'{page}: expected exactly one Notebook 06 draft audit by source_md5, found {len(drafts)}')
    require(len(reviews)<=1,f'{page}: duplicate Notebook 07 review audits by source_md5')
    d=drafts[0]; r=reviews[0] if reviews else None
    draft_mask,_=existing_path(d,['draft_mask','draft_mask_path'],'draft')
    confidence,_=existing_path(d,['confidence_map','confidence_map_path'],'draft')
    reviewed_mask=None; aligned=source
    if r is not None:
        reviewed_mask,_=existing_path(r,['corrected_mask_output','corrected_mask','mask_output'],'review')
        try: aligned,_=existing_path(r,['aligned_output','aligned_page','observed_output'],'review')
        except Exception: aligned=source
    resolved.append({'page':page,'source_path':str(source),'source_md5':smd5,'aligned_path':str(aligned),'draft_mask_path':str(draft_mask),'confidence_path':str(confidence),'draft_audit_path':d['_audit_path'],'reviewed_mask_path':str(reviewed_mask) if reviewed_mask else None,'review_audit_path':r['_audit_path'] if r else None,'draft_audit_json':json.dumps(d,sort_keys=True,default=str),'review_audit_json':json.dumps(r,sort_keys=True,default=str) if r else None})
resolution=pd.DataFrame(resolved).sort_values('page'); resolution.to_csv(AUDIT/'hitl_input_resolution.csv',index=False)
print('Resolved pages:',len(resolution),'with review:',resolution.reviewed_mask_path.notna().sum())

## 5. Resolve the established inherited YOLO confidence-gating policy

In [ ]:
# PROVENANCE: NEW for V9.0A; purpose: lift the established gate from prior Notebook 06 audit evidence and refuse threshold invention.
GATE_KEYS=['confidence_gate','confidence_threshold','mask_confidence_threshold','pixel_confidence_threshold','established_confidence_gate','prediction_confidence']
POLICY_KEYS=['confidence_gate_policy_id','gate_policy_id','threshold_policy_id','policy_id','detector_policy_id']

def find_numeric(record,keys):
    for key in keys:
        if key in record and record[key] not in [None,'']:
            try:
                value=float(record[key])
                if 0<=value<=1:return value,key
            except Exception: pass
    for container in ['settings','inference','configuration','config','policy']:
        child=record.get(container)
        if isinstance(child,dict):
            found=find_numeric(child,keys)
            if found:return found
    return None

def find_text(record,keys):
    for key in keys:
        if key in record and record[key] not in [None,'']: return str(record[key]),key
    for container in ['settings','inference','configuration','config','policy']:
        child=record.get(container)
        if isinstance(child,dict):
            found=find_text(child,keys)
            if found:return found
    return None

candidates=[]
for _,row in resolution.iterrows():
    record=json.loads(row.draft_audit_json); g=find_numeric(record,GATE_KEYS); pid=find_text(record,POLICY_KEYS)
    candidates.append({'page':row.page,'gate':g[0] if g else np.nan,'gate_field':g[1] if g else None,'policy_id':pid[0] if pid else None,'policy_field':pid[1] if pid else None,'audit_path':row.draft_audit_path})
gates=pd.DataFrame(candidates); gates.to_csv(AUDIT/'confidence_gate_resolution.csv',index=False)
unique=sorted(gates.gate.dropna().unique().tolist())
if len(unique)==1:
    ESTABLISHED_GATE=float(unique[0]); gate_resolution='draft_audit_consensus'
elif ALLOW_MANUAL_GATE_OVERRIDE and ESTABLISHED_GATE_OVERRIDE is not None and ESTABLISHED_GATE_POLICY_ID_OVERRIDE:
    ESTABLISHED_GATE=float(ESTABLISHED_GATE_OVERRIDE); gate_resolution='explicit_inherited_policy_override'
else:
    raise RuntimeError(f'Established YOLO gate did not resolve uniquely from Notebook 06 audits. Values={unique}. Supply a documented prior-policy override only by enabling ALLOW_MANUAL_GATE_OVERRIDE.')
require(0<=ESTABLISHED_GATE<=1,'Resolved gate outside [0,1].')
policy_ids=sorted(x for x in gates.policy_id.dropna().unique().tolist())
GATE_POLICY_ID=(policy_ids[0] if len(policy_ids)==1 else ESTABLISHED_GATE_POLICY_ID_OVERRIDE)
require(GATE_POLICY_ID is not None,'Established gate requires a policy/detector identity, not only a number.')
gate_record={'gate':ESTABLISHED_GATE,'policy_id':GATE_POLICY_ID,'resolution':gate_resolution,'source_audits':gates.audit_path.tolist(),'not_optimised_in_v9_0a':True}
atomic_json(AUDIT/'established_yolo_gate.json',gate_record)
print(json.dumps(gate_record,indent=2))

## 6. Build and lock the four mask-authority arms

In [ ]:
# PROVENANCE: NEW for V9.0A, using Notebook 06 confidence-map semantics and Notebook 07 corrected-mask lineage; purpose: construct paired mask arms without changing V8.8.
arm_rows=[]; edit_rows=[]
for _,r in resolution.iterrows():
    page=r.page; observed=read_gray(r.aligned_path); raw=binary(read_gray(r.draft_mask_path)); conf=np.load(r.confidence_path).astype(np.float32)
    require(conf.shape==observed.shape==raw.shape,f'{page}: observed/raw/confidence shape mismatch')
    require(np.isfinite(conf).all(),f'{page}: non-finite confidence values')
    reviewed=binary(read_gray(r.reviewed_mask_path)) if pd.notna(r.reviewed_mask_path) and r.reviewed_mask_path else None
    require(reviewed is not None,f'{page}: reviewed comparator is required for all V9.0A pages')
    require(reviewed.shape==observed.shape,f'{page}: reviewed mask shape mismatch')
    gated=np.where((raw>0)&(conf>=ESTABLISHED_GATE),255,0).astype(np.uint8)
    arms={'reviewed_comparator':reviewed,'raw_yolo':raw,'confidence_gated_yolo':gated,'reviewer_corrected_yolo':reviewed.copy()}
    additions=(reviewed>0)&(raw==0); erasures=(reviewed==0)&(raw>0)
    edit_rows.append({'page':page,'raw_pixels':int((raw>0).sum()),'reviewed_pixels':int((reviewed>0).sum()),'reviewer_additions':int(additions.sum()),'reviewer_erasures':int(erasures.sum()),'raw_reviewed_xor':int(np.logical_xor(raw>0,reviewed>0).sum())})
    for arm,mask in arms.items():
        d=MASKS/arm; d.mkdir(parents=True,exist_ok=True); path=d/f'{page}.png'; require(cv2.imwrite(str(path),mask),f'{page}/{arm}: mask write failed')
        arm_rows.append({'page':page,'arm':arm,'mask_path':str(path),'mask_pixels':int((mask>0).sum()),'mask_array_sha256':array_hash(mask),'mask_file_sha256':sha256_file(path),'observed_path':r.aligned_path,'observed_sha256':sha256_file(r.aligned_path),'confidence_path':r.confidence_path,'confidence_sha256':sha256_file(r.confidence_path),'gate':ESTABLISHED_GATE if arm=='confidence_gated_yolo' else np.nan,'gate_policy_id':GATE_POLICY_ID if arm=='confidence_gated_yolo' else None})
arm_manifest=pd.DataFrame(arm_rows).sort_values(['arm','page']); arm_manifest.to_csv(LOCK/'mask_arm_manifest.csv',index=False)
pd.DataFrame(edit_rows).to_csv(SCORE/'reviewer_edits.csv',index=False)
mask_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'roster':sorted(resolution.page.tolist()),'arms':ARMS,'manifest_sha256':sha256_file(LOCK/'mask_arm_manifest.csv'),'gate_record_sha256':sha256_file(AUDIT/'established_yolo_gate.json'),'v88_lineage_sha256':sha256_file(AUDIT/'v88_lineage.json'),'truth_loaded':False}
mask_lock['lock_id']=object_hash(mask_lock); atomic_json(LOCK/'mask_lock.json',mask_lock)
print('Mask lock:',mask_lock['lock_id'])

## 7. Execute the frozen V8.8 engine for every arm

In [ ]:
# PROVENANCE: NEW V9.0A orchestration around Notebook 53R3 durable implementation; purpose: require arm-specific frozen-engine outputs with exact source identity rather than inventing a callable API.
#
# IMPORTANT EXECUTION CONTRACT
# Notebook 53R3 is the canonical executable engine, but its historical run consumes a specific frozen input roster.
# V9.0A therefore requires one completed Notebook 53R3-compatible run per arm, produced by supplying that arm's masks through
# the same canonical input adapter used for the historical 53R3 run. This cell resolves and validates those runs.
# It does not search for or invent a Python module/API.
ARM_RUN_OVERRIDES={arm:None for arm in ARMS}
ARM_RUN_ROOTS={arm:ENGINE/arm for arm in ARMS}
REQUIRED_ARM_OUTPUTS=['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','tables/mask_inventory.csv','pages']

arm_runs={}; run_rows=[]
for arm in ARMS:
    run=Path(ARM_RUN_OVERRIDES[arm]) if ARM_RUN_OVERRIDES[arm] else latest_complete(ARM_RUN_ROOTS[arm],REQUIRED_ARM_OUTPUTS)
    if run is None:
        template=ENGINE/arm/'README_REQUIRED_RUN.txt'; template.parent.mkdir(parents=True,exist_ok=True)
        template.write_text('Create a Notebook 53R3-compatible run for this exact V9.0A arm using 02_mask_arms/'+arm+' as the authoritative seed-mask roster. Preserve the frozen Notebook 53R3 implementation and write its standard stage_f_verdict.json, experiment_identity.json, tables and pages outputs here. Do not edit thresholds or reconstruction logic.')
        raise RuntimeError(f'Missing completed frozen V8.8 run for arm {arm}. See {template}')
    identity=read_json(run/'experiment_identity.json'); verdict=read_json(run/'stage_f_verdict.json')
    require(identity.get('version')=='53R3-v8.8b-notebook51R2-regression-compatibility',f'{arm}: wrong V8.8 engine version')
    require(identity.get('mode')=='REGISTERED_REPLAY',f'{arm}: wrong V8.8 mode')
    page_dirs={p.name:p for p in (run/'pages').iterdir() if p.is_dir() and (p/'corrected.png').exists()}
    require(set(page_dirs)==set(resolution.page),f'{arm}: V8.8 output roster mismatch')
    arm_runs[arm]=run
    for page,pdir in page_dirs.items():
        corrected=read_gray(pdir/'corrected.png'); mask=read_gray(MASKS/arm/f'{page}.png'); observed=read_gray(resolution.set_index('page').loc[page,'aligned_path'])
        require(corrected.shape==mask.shape==observed.shape,f'{arm}/{page}: output shape mismatch')
        require(np.array_equal(corrected[mask==0],observed[mask==0]),f'{arm}/{page}: outside-authority drift')
        run_rows.append({'arm':arm,'page':page,'run':str(run),'corrected_path':str(pdir/'corrected.png'),'corrected_sha256':sha256_file(pdir/'corrected.png'),'corrected_array_sha256':array_hash(corrected),'outside_authority_drift':0,'engine_version':identity.get('version'),'engine_mode':identity.get('mode')})
engine_outputs=pd.DataFrame(run_rows).sort_values(['arm','page']); engine_outputs.to_csv(LOCK/'v88_output_manifest.csv',index=False)
output_lock={'created_utc':datetime.now(timezone.utc).isoformat(),'mask_lock_id':mask_lock['lock_id'],'v88_output_manifest_sha256':sha256_file(LOCK/'v88_output_manifest.csv'),'arm_runs':{k:str(v) for k,v in arm_runs.items()},'truth_loaded':False}
output_lock['lock_id']=object_hash(output_lock); atomic_json(LOCK/'prediction_and_output_lock.json',output_lock)
print('Prediction/output lock:',output_lock['lock_id'])

## 8. Post-lock mask-authority and output comparisons

In [ ]:
# PROVENANCE: NEW for V9.0A; purpose: calculate paired mask/output transfer metrics only after mask and page outputs are locked.
def binary_metrics(candidate,reference):
    c=candidate>0; r=reference>0; inter=int((c&r).sum()); union=int((c|r).sum()); tp=inter; fp=int((c&~r).sum()); fn=int((~c&r).sum())
    return {'intersection':inter,'union':union,'iou':inter/max(union,1),'precision_vs_reviewed':tp/max(tp+fp,1),'recall_vs_reviewed':tp/max(tp+fn,1),'false_positive_pixels_vs_reviewed':fp,'false_negative_pixels_vs_reviewed':fn,'xor_pixels':int(np.logical_xor(c,r).sum())}
rows=[]; output_rows=[]
for page in sorted(resolution.page):
    ref=read_gray(MASKS/'reviewed_comparator'/f'{page}.png')
    ref_out=read_gray(Path(engine_outputs[(engine_outputs.arm=='reviewed_comparator')&(engine_outputs.page==page)].iloc[0].corrected_path))
    for arm in ARMS:
        mask=read_gray(MASKS/arm/f'{page}.png'); q=binary_metrics(mask,ref); rows.append({'page':page,'arm':arm,**q})
        out=read_gray(Path(engine_outputs[(engine_outputs.arm==arm)&(engine_outputs.page==page)].iloc[0].corrected_path)); diff=out!=ref_out
        output_rows.append({'page':page,'arm':arm,'corrected_xor_pixels_vs_reviewed_arm':int(diff.sum()),'corrected_mae_vs_reviewed_arm':float(np.abs(out.astype(np.int16)-ref_out.astype(np.int16)).mean()),'corrected_max_abs_difference_vs_reviewed_arm':int(np.abs(out.astype(np.int16)-ref_out.astype(np.int16)).max())})
mask_metrics=pd.DataFrame(rows); output_metrics=pd.DataFrame(output_rows)
mask_metrics.to_csv(SCORE/'mask_metrics_vs_reviewed.csv',index=False); output_metrics.to_csv(SCORE/'corrected_output_metrics_vs_reviewed_arm.csv',index=False)
summary=mask_metrics.merge(output_metrics,on=['page','arm']).groupby('arm',as_index=False).agg(pages=('page','nunique'),mask_iou_mean=('iou','mean'),mask_precision_mean=('precision_vs_reviewed','mean'),mask_recall_mean=('recall_vs_reviewed','mean'),mask_xor_pixels=('xor_pixels','sum'),corrected_xor_pixels=('corrected_xor_pixels_vs_reviewed_arm','sum'),corrected_mae_mean=('corrected_mae_vs_reviewed_arm','mean'))
summary.to_csv(SCORE/'arm_summary_vs_reviewed.csv',index=False); display(summary)

## 9. Optional clean-truth scoring after lock

In [ ]:
# PROVENANCE: TRANSPLANTED and adapted from the project truth-firewall pattern in Notebooks 10, 17 and 28; purpose: load clean truth only after frozen mask/output locks exist.
require((LOCK/'mask_lock.json').exists() and (LOCK/'prediction_and_output_lock.json').exists(),'Locks must exist before truth scoring.')
clean_map=image_map(CLEAN_DIR); missing=sorted(set(resolution.page)-set(clean_map))
truth_available=len(missing)==0
truth_rows=[]
if truth_available:
    for page in sorted(resolution.page):
        clean=read_gray(clean_map[page])
        for arm in ARMS:
            out=read_gray(Path(engine_outputs[(engine_outputs.arm==arm)&(engine_outputs.page==page)].iloc[0].corrected_path))
            require(clean.shape==out.shape,f'{page}: clean/output shape mismatch')
            diff=np.abs(clean.astype(np.float32)-out.astype(np.float32)); mse=float(np.mean((clean.astype(np.float32)-out.astype(np.float32))**2))
            truth_rows.append({'page':page,'arm':arm,'whole_page_mae':float(diff.mean()),'whole_page_rmse':float(np.sqrt(mse))})
    pd.DataFrame(truth_rows).to_csv(SCORE/'clean_truth_whole_page_metrics.csv',index=False)
else:
    pd.DataFrame({'missing_clean_page':missing}).to_csv(AUDIT/'clean_truth_unavailable.csv',index=False)
print('Clean truth scoring available:',truth_available)

## 10. Failure taxonomy and visual audit package

In [ ]:
# PROVENANCE: NEW for V9.0A, following project visual-audit conventions; purpose: localise mask and corrected-page disagreements without changing decisions.
queue=[]
for page in sorted(resolution.page):
    reviewed=read_gray(MASKS/'reviewed_comparator'/f'{page}.png')>0
    raw=read_gray(MASKS/'raw_yolo'/f'{page}.png')>0
    gated=read_gray(MASKS/'confidence_gated_yolo'/f'{page}.png')>0
    additions=reviewed&~raw; erasures=raw&~reviewed; gate_removed=raw&~gated
    queue.append({'page':page,'reviewer_additions':int(additions.sum()),'reviewer_erasures':int(erasures.sum()),'gate_removed_raw_pixels':int(gate_removed.sum()),'priority_pixels':int((additions|erasures|gate_removed).sum())})
queue=pd.DataFrame(queue).sort_values('priority_pixels',ascending=False); queue.to_csv(SCORE/'visual_audit_queue.csv',index=False)
for page in queue.head(30).page:
    obs=read_gray(resolution.set_index('page').loc[page,'aligned_path']); rgb=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR)
    reviewed=read_gray(MASKS/'reviewed_comparator'/f'{page}.png')>0; raw=read_gray(MASKS/'raw_yolo'/f'{page}.png')>0; gated=read_gray(MASKS/'confidence_gated_yolo'/f'{page}.png')>0
    overlay=rgb.copy(); overlay[reviewed&~raw]=(0,200,0); overlay[raw&~reviewed]=(0,0,255); overlay[raw&~gated]=(0,191,255)
    out=cv2.addWeighted(rgb,.55,overlay,.45,0); cv2.imwrite(str(VISUAL/f'{page}_mask_disagreement.png'),out)
print('Visual audit queue:',SCORE/'visual_audit_queue.csv')

## 11. Fail-closed verdict

In [ ]:
# PROVENANCE: NEW for V9.0A; purpose: issue a narrow evidence verdict without granting automatic authority.
checks={
 'v88_lineage_verified':(AUDIT/'v88_lineage.json').exists(),
 'established_gate_resolved':(AUDIT/'established_yolo_gate.json').exists(),
 'all_pages_have_reviewed_comparator':bool(resolution.reviewed_mask_path.notna().all()),
 'four_arm_roster_complete':bool(len(arm_manifest)==len(resolution)*len(ARMS)),
 'mask_lock_written':(LOCK/'mask_lock.json').exists(),
 'v88_outputs_locked':(LOCK/'prediction_and_output_lock.json').exists(),
 'zero_outside_authority_drift':bool((engine_outputs.outside_authority_drift==0).all()),
 'reviewer_edits_recorded':(SCORE/'reviewer_edits.csv').exists(),
 'comparisons_post_lock':(SCORE/'arm_summary_vs_reviewed.csv').exists(),
}
first_failure=next((k for k,v in checks.items() if not v),None)
verdict='PASS_V9_0A_EVIDENCE_PACKAGE_READY_FOR_REVIEW' if first_failure is None else 'FAIL_V9_0A_INVALID_RUN'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-mask-authority-transfer','verdict':verdict,'first_failed_gate':first_failure,'checks':checks,'mask_lock_id':mask_lock['lock_id'],'prediction_output_lock_id':output_lock['lock_id'],'automatic_mask_authority_authorised':False,'production_output_authority':False,'reviewed_mask_is_comparator_not_perfect_truth':True,'gate_optimised_in_v9_0a':False}
atomic_json(OUT/'v9_0a_verdict.json',record)
print(json.dumps(record,indent=2))
if first_failure is not None: raise RuntimeError(f'V9.0A failed at {first_failure}')

## 12. Standard Final Upload Package

In [ ]:
# PROVENANCE: TRANSPLANTED and adapted from the standard Final Upload Package requirement introduced in the V8.2 decision record; purpose: curate review evidence with hashes.
UPLOAD_ARTEFACTS=[
 OUT/'v9_0a_verdict.json', AUDIT/'v88_lineage.json', AUDIT/'established_yolo_gate.json', AUDIT/'hitl_input_resolution.csv', AUDIT/'confidence_gate_resolution.csv',
 LOCK/'mask_lock.json', LOCK/'mask_arm_manifest.csv', LOCK/'prediction_and_output_lock.json', LOCK/'v88_output_manifest.csv',
 SCORE/'reviewer_edits.csv', SCORE/'mask_metrics_vs_reviewed.csv', SCORE/'corrected_output_metrics_vs_reviewed_arm.csv', SCORE/'arm_summary_vs_reviewed.csv', SCORE/'visual_audit_queue.csv'
]
if (SCORE/'clean_truth_whole_page_metrics.csv').exists(): UPLOAD_ARTEFACTS.append(SCORE/'clean_truth_whole_page_metrics.csv')
UPLOAD_ARTEFACTS += sorted(VISUAL.glob('*.png'))[:30]
missing=[str(p) for p in UPLOAD_ARTEFACTS if not p.exists()]
require(not missing,f'Upload package missing required artefacts: {missing}')
rows=[]
for src in UPLOAD_ARTEFACTS:
    rel=src.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
manifest=pd.DataFrame(rows).sort_values('relative_path'); manifest.to_csv(UPLOAD/'upload_manifest.csv',index=False)
readme='V9.0A MASK-AUTHORITY TRANSFER REVIEW PACKAGE\n\nResearch evidence only. No automatic mask authority or production output is authorised.\nStart with v9_0a_verdict.json, 05_scoring/arm_summary_vs_reviewed.csv, 05_scoring/visual_audit_queue.csv and 07_audit/established_yolo_gate.json.\n'
(UPLOAD/'README.txt').write_text(readme)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(manifest.relative_path.tolist()))
zip_path=UPLOAD/'58_V9_0A_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print('Final Upload Package:',zip_path)

## Interpretation

A PASS from this notebook means only that the paired mask-authority experiment is internally complete and ready for review. It does not mean that raw or confidence-gated YOLO masks are authorised automatically. Any future authority tier must be frozen in a separate decision record with predeclared acceptance gates, source-group and hard-negative evidence, and supervised exception handling.